# slice routing judge: LoRA training (phase 20)

This notebook trains a small LoRA judge on slice's own routed traffic so we can
retire the rented Haiku judge that currently labels each request easy or hard.

The judge learns from real routed traffic: prompts that Haiku served are labeled
`easy`, prompts that Sonnet served are labeled `hard`. A tiny fine tuned model
that reproduces those routing decisions locally removes a per request call to a
rented model.

Runs top to bottom on a free Colab T4. It downloads one base model from Hugging
Face and makes no other network calls. It never contacts the slice gateway and
uses no secrets. Printed output is counts and metrics only, never prompt text.

## 1. Installs

Latest Colab wheels work as of this writing, so nothing is pinned. If a future
Colab image breaks an import, pin the offending package here (for example
`transformers==4.44.2`) and rerun this cell only.

In [ ]:
%pip install -q transformers peft datasets accelerate
# Colab's preinstalled torchao 0.10 trips peft's import check; remove it.
%pip uninstall -y torchao

## 2. Upload the data

Upload `judge_train.jsonl` and `judge_eval.jsonl` when prompted. These files are
produced by `scripts/prepare_judge_data.py` and are never committed to the repo.
Only row counts are printed.

In [ ]:
import os
from google.colab import files

uploaded = files.upload()  # select judge_train.jsonl and judge_eval.jsonl

TRAIN_PATH = 'judge_train.jsonl'
EVAL_PATH = 'judge_eval.jsonl'

for path in (TRAIN_PATH, EVAL_PATH):
    assert os.path.exists(path), f'missing {path}, upload it in the cell above'

def count_rows(path):
    with open(path, 'r', encoding='utf-8') as handle:
        return sum(1 for line in handle if line.strip())

print('judge_train.jsonl rows:', count_rows(TRAIN_PATH))
print('judge_eval.jsonl rows:', count_rows(EVAL_PATH))

## 3. Formatting

Each row becomes one chat sample. The system message fixes the task, the user
message carries the prompt field, and the assistant message is the label. The
same system message is reused at eval and inference time.

In [ ]:
from datasets import load_dataset

SYSTEM_MESSAGE = (
    "You are slice's routing judge. "
    'Answer with exactly one word, easy or hard.'
)
LABELS = ('easy', 'hard')

def build_messages(prompt, label=None):
    messages = [
        {'role': 'system', 'content': SYSTEM_MESSAGE},
        {'role': 'user', 'content': prompt},
    ]
    if label is not None:
        messages.append({'role': 'assistant', 'content': label})
    return messages

train_raw = load_dataset('json', data_files=TRAIN_PATH, split='train')
eval_raw = load_dataset('json', data_files=EVAL_PATH, split='train')

# Sanity check on labels only. No prompt text is printed.
for split_name, ds in (('train', train_raw), ('eval', eval_raw)):
    bad = [x for x in set(ds['label']) if x not in LABELS]
    assert not bad, f'unexpected labels in {split_name}: {bad}'
print('train rows:', len(train_raw), ' eval rows:', len(eval_raw))

## 4. Model and LoRA

Base model is `Qwen/Qwen2.5-0.5B-Instruct`. It loads in fp16 because the T4 is a
Turing card without bf16. LoRA adapts the attention projections with r 16, alpha
32, dropout 0.05. Gradient checkpointing is on and every seed is 42.

In [ ]:
import random
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, get_peft_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,  # T4 is Turing, fp16 not bf16
    device_map='cuda',
)
model.config.use_cache = False  # required with gradient checkpointing
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Tokenize with completion masking and train

Labels are masked to `-100` on the prompt tokens, so the causal loss lands only
on the answer tokens. Three epochs, effective batch size 16 (8 per device times 2
accumulation steps), which fits comfortably in 16 GB for a 0.5B model in fp16.

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

MAX_LEN = 1024
MAX_PROMPT_TOKENS = 768  # keep room for the chat scaffold and the answer

def truncate_prompt(prompt):
    ids = tokenizer(prompt, add_special_tokens=False).input_ids
    if len(ids) > MAX_PROMPT_TOKENS:
        prompt = tokenizer.decode(ids[:MAX_PROMPT_TOKENS])
    return prompt

def tokenize_train(example):
    prompt = truncate_prompt(example['prompt'])
    full_ids = tokenizer.apply_chat_template(
        build_messages(prompt, example['label']),
        tokenize=True, return_dict=False, add_generation_prompt=False,
    )
    prompt_ids = tokenizer.apply_chat_template(
        build_messages(prompt),
        tokenize=True, return_dict=False, add_generation_prompt=True,
    )
    full_ids = full_ids[:MAX_LEN]
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    labels = labels[:len(full_ids)]
    return {
        'input_ids': full_ids,
        'attention_mask': [1] * len(full_ids),
        'labels': labels,
    }

train_ds = train_raw.map(tokenize_train, remove_columns=train_raw.column_names)

collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, label_pad_token_id=-100, padding=True,
)

args = TrainingArguments(
    output_dir='judge_out',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=4,
    logging_steps=10,
    save_strategy='no',
    report_to='none',
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    seed=SEED,
    data_seed=SEED,
    optim='adamw_torch',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=collator,
)
trainer.train()

## 6. Evaluate (no sampling)

For each eval row we score the two candidate answers under the same chat template
and pick the higher scoring one. When a candidate word is more than one token, we
compare the summed token logprobs. We report overall accuracy, accuracy per
label, confusion counts, and accuracy split by whether `template_category` agrees
with the label. The 58.1 percent majority baseline is printed for context.

In [ ]:
import torch.nn.functional as F
from collections import Counter

model.config.use_cache = True
model.gradient_checkpointing_disable()
model.eval()

MAJORITY_BASELINE = 0.581

# Precompute candidate token ids once.
CANDIDATE_IDS = {w: tokenizer(w, add_special_tokens=False).input_ids for w in LABELS}

@torch.no_grad()
def decision(prompt):
    prompt = truncate_prompt(prompt)
    prompt_ids = tokenizer.apply_chat_template(
        build_messages(prompt), tokenize=True, return_dict=False, add_generation_prompt=True,
    )
    scores = {}
    for word, word_ids in CANDIDATE_IDS.items():
        input_ids = torch.tensor([prompt_ids + word_ids], device=model.device)
        logits = model(input_ids).logits[0]
        logprobs = F.log_softmax(logits.float(), dim=-1)
        start = len(prompt_ids)
        total = 0.0
        for j, tok in enumerate(word_ids):
            total += logprobs[start + j - 1, tok].item()  # summed logprobs
        scores[word] = total
    return max(scores, key=scores.get)

correct = 0
per_label_total = Counter()
per_label_correct = Counter()
confusion = Counter()  # (true, pred)
agree_total = agree_correct = 0
disagree_total = disagree_correct = 0

for row in eval_raw:
    true = row['label']
    pred = decision(row['prompt'])
    hit = int(pred == true)
    correct += hit
    per_label_total[true] += 1
    per_label_correct[true] += hit
    confusion[(true, pred)] += 1
    agrees = row.get('template_category') == true
    if agrees:
        agree_total += 1
        agree_correct += hit
    else:
        disagree_total += 1
        disagree_correct += hit

n = len(eval_raw)
def pct(num, den):
    return (100.0 * num / den) if den else float('nan')

print('overall accuracy: {:.1f}% ({}/{})'.format(pct(correct, n), correct, n))
print('majority baseline: {:.1f}%  (58.1% reference)'.format(100 * MAJORITY_BASELINE))
print()
print('accuracy per label:')
for label in LABELS:
    print('  {}: {:.1f}% ({}/{})'.format(
        label, pct(per_label_correct[label], per_label_total[label]),
        per_label_correct[label], per_label_total[label]))
print()
print('confusion counts (true -> pred):')
for true in LABELS:
    for pred in LABELS:
        print('  {} -> {}: {}'.format(true, pred, confusion[(true, pred)]))
print()
print('accuracy where template_category agrees with label: {:.1f}% ({}/{})'.format(
    pct(agree_correct, agree_total), agree_correct, agree_total))
print('accuracy where template_category disagrees with label: {:.1f}% ({}/{})'.format(
    pct(disagree_correct, disagree_total), disagree_correct, disagree_total))

## 7. Latency

Single decision latency on the T4, measured over the eval set after a short
warmup. This is the before number for the later TensorRT-LLM conversion.

In [ ]:
import time

warmup_prompts = [eval_raw[i]['prompt'] for i in range(min(5, len(eval_raw)))]
for p in warmup_prompts:
    decision(p)
torch.cuda.synchronize()

timings_ms = []
for row in eval_raw:
    torch.cuda.synchronize()
    start = time.perf_counter()
    decision(row['prompt'])
    torch.cuda.synchronize()
    timings_ms.append((time.perf_counter() - start) * 1000.0)

timings_ms.sort()
mean_ms = sum(timings_ms) / len(timings_ms)
p95_ms = timings_ms[min(len(timings_ms) - 1, int(round(0.95 * (len(timings_ms) - 1))))]
print('single decision latency over {} eval rows'.format(len(timings_ms)))
print('  mean: {:.1f} ms'.format(mean_ms))
print('  p95:  {:.1f} ms'.format(p95_ms))

## 8. Save and download

Saves the LoRA adapter to `judge_lora/` and the merged full model to
`judge_merged/`, zips both, and offers download links. The adapter is small and
rides on top of the base model; the merged model is the standalone artifact for
TensorRT-LLM conversion.

In [ ]:
import shutil

# Adapter only.
model.save_pretrained('judge_lora')
tokenizer.save_pretrained('judge_lora')

# Merged standalone model for TensorRT-LLM. merge_and_unload folds the adapter in.
merged = model.merge_and_unload()
merged.save_pretrained('judge_merged')
tokenizer.save_pretrained('judge_merged')

lora_zip = shutil.make_archive('judge_lora', 'zip', 'judge_lora')
merged_zip = shutil.make_archive('judge_merged', 'zip', 'judge_merged')
print('wrote', os.path.basename(lora_zip))
print('wrote', os.path.basename(merged_zip))

files.download('judge_lora.zip')
files.download('judge_merged.zip')

## 9. Fresh eval on unseen templates

`judge_eval_fresh.jsonl` comes from `scripts/generate_fresh_eval.py`: 100
prompts drawn from templates the judge never saw in training, sent through
the live gateway and labeled by the model that served them. This is the
reported number for the judge, since it measures generalization rather than
memorized templates. Upload the file when prompted; counts only are printed.

In [ ]:
# Fresh eval: 100 prompts from templates the judge never saw.
from collections import Counter

model = merged
model.eval()

up = files.upload()  # select judge_eval_fresh.jsonl
FRESH_PATH = 'judge_eval_fresh.jsonl'
assert os.path.exists(FRESH_PATH), 'missing judge_eval_fresh.jsonl'
fresh_raw = load_dataset('json', data_files=FRESH_PATH, split='train')

n = len(fresh_raw)
label_counts = Counter(fresh_raw['label'])
majority = max(label_counts.values()) / n

correct = 0
per_label_total = Counter()
per_label_correct = Counter()
confusion = Counter()
agree_total = agree_correct = 0
disagree_total = disagree_correct = 0

for row in fresh_raw:
    true = row['label']
    pred = decision(row['prompt'])
    hit = int(pred == true)
    correct += hit
    per_label_total[true] += 1
    per_label_correct[true] += hit
    confusion[(true, pred)] += 1
    if row.get('template_category') == true:
        agree_total += 1
        agree_correct += hit
    else:
        disagree_total += 1
        disagree_correct += hit

print('FRESH EVAL, unseen templates')
print('rows:', n, ' label counts:', dict(label_counts))
print('overall accuracy: {:.1f}% ({}/{})'.format(pct(correct, n), correct, n))
print('majority baseline: {:.1f}%'.format(100 * majority))
print()
print('accuracy per label:')
for label in LABELS:
    print('  {}: {:.1f}% ({}/{})'.format(
        label, pct(per_label_correct[label], per_label_total[label]),
        per_label_correct[label], per_label_total[label]))
print()
print('confusion counts (true -> pred):')
for true in LABELS:
    for pred in LABELS:
        print('  {} -> {}: {}'.format(true, pred, confusion[(true, pred)]))
print()
print('accuracy where template_category agrees with label: {:.1f}% ({}/{})'.format(
    pct(agree_correct, agree_total), agree_correct, agree_total))
print('accuracy where template_category disagrees with label: {:.1f}% ({}/{})'.format(
    pct(disagree_correct, disagree_total), disagree_correct, disagree_total))